In [2]:
import os
import random
import string
import math

import hail as hl
from ukb_utils import hail_init
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
from ukb_utils import genotypes
from ko_utils import ko
from ko_utils import io
from ko_utils import samples
from ko_utils.ko import PLOF_CSQS, MISSENSE_CSQS, SYNONYMOUS_CSQS, OTHER_CSQS
import scipy.stats as stats

In [3]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Initializing Hail on SLURM with 1 CPU(s).


2022-12-21 12:54:23 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Running on Apache Spark version 3.1.3
SparkUI available at http://compa041.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.102-817f6fb3468f
LOGGING: writing to logs/hail/hail_test_export.log


In [4]:
imp_path = "data/unphased/imputed/common_new/ukb_imp_200k_common_chr21.mt"
imp = hl.read_matrix_table(imp_path)

In [5]:
calls_path = "data/unphased/calls/liftover/ukb_liftover_calls_500k_chr21.mt/"
calls = hl.read_matrix_table(calls_path)

In [16]:
calls.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'ref_allele_mismatch': bool
    'locus_fail_liftover': bool
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [13]:
imp.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'varid': str
    'info_score': float64
    'new_locus': struct {
        result: locus<GRCh38>, 
        is_negative_strand: bool
    }
    'old_locus': locus<GRCh37>
    'new_alleles': array<str>
    'ref_allele_mismatch': bool
    'locus_fail_liftover': bool
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [15]:
calls.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'ref_allele_mismatch': bool
    'locus_fail_liftover': bool
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [29]:
hl.missing(hl.call())

TypeError: missing: parameter 't': expected (hail.expr.types.HailType or (str) or NoneType), found hail.expr.expressions.typed_expressions.CallExpression: <CallExpression of type call>

In [36]:
hl.call().show()

""
""
call
-


In [ ]:
ht_ko = hl.import_table("data/phenotypes/samples/sample_list_phased.txt", no_header = True, key = 'f0')
ht_qc = hl.import_table("data/phenotypes/samples/sample_list_final.qc.txt", no_header = True, key = 'f0')

In [19]:
calls = calls.filter_cols(hl.is_defined(ht_phased[calls.col_key]))
calls = calls.filter_cols(hl.is_defined(ht_qc[calls.col_key]))
calls = calls.filter_cols(~hl.is_defined(imp.cols()[calls.col_key]))

In [20]:
calls.count_cols()

FatalError: OutOfMemoryError: Java heap space

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 11.0 failed 1 times, most recent failure: Lost task 0.0 in stage 11.0 (TID 11) (compa041.hpc.in.bmrc.ox.ac.uk executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.util.Arrays.copyOf(Arrays.java:3236)
	at java.io.ByteArrayOutputStream.grow(ByteArrayOutputStream.java:118)
	at java.io.ByteArrayOutputStream.ensureCapacity(ByteArrayOutputStream.java:93)
	at java.io.ByteArrayOutputStream.write(ByteArrayOutputStream.java:153)
	at org.apache.spark.util.ByteBufferOutputStream.write(ByteBufferOutputStream.scala:41)
	at java.io.ObjectOutputStream$BlockDataOutputStream.write(ObjectOutputStream.java:1853)
	at java.io.ObjectOutputStream.write(ObjectOutputStream.java:709)
	at org.apache.spark.util.Utils$.writeByteBuffer(Utils.scala:233)
	at org.apache.spark.scheduler.DirectTaskResult.$anonfun$writeExternal$1(TaskResult.scala:53)
	at org.

In [49]:
input_path = "data/unphased/imputed/common_8cores/ukb_imp_200k_common_chr20.mt"

In [50]:
mt = hl.read_matrix_table(input_path)
mt.count_cols()

198895

In [35]:
# what are the sample overlaps
imputed = genotypes.get_ukb_imputed_v3_bgen([21])
calls = genotypes.get_ukb_genotypes_bed([21])
cols_imputed = imputed.s.collect()
cols_calls = calls.s.collect()
print(len(cols_imputed))
print(len(cols_calls))

ht = hl.import_table("data/knockouts/alt/samples.txt", no_header = True)
cols_ko = ht.f0.collect()

print(len(cols_ko))
print(len(list(set(cols_imputed) & set(cols_ko))))
print(len(list(set(cols_calls) & set(cols_ko))))
print(len(list(set(cols_calls) & set(cols_ko) & set(cols_ko))))

2022-12-21 08:49:39 Hail: INFO: Number of BGEN files parsed: 1
2022-12-21 08:49:39 Hail: INFO: Number of samples in BGEN files: 487409
2022-12-21 08:49:39 Hail: INFO: Number of variants across all BGEN files: 1261158
2022-12-21 08:50:06 Hail: INFO: Found 488377 samples in fam file.
2022-12-21 08:50:06 Hail: INFO: Found 11342 variants in bim file.


487409
488377


2022-12-21 08:50:55 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)


176587
176266
176587
176587


In [36]:
imputed = genotypes.get_ukb_imputed_v3_bgen([21])
calls = genotypes.get_ukb_genotypes_bed([21])

2022-12-21 09:03:19 Hail: INFO: Number of BGEN files parsed: 1
2022-12-21 09:03:19 Hail: INFO: Number of samples in BGEN files: 487409
2022-12-21 09:03:19 Hail: INFO: Number of variants across all BGEN files: 1261158
2022-12-21 09:03:47 Hail: INFO: Found 488377 samples in fam file.
2022-12-21 09:03:47 Hail: INFO: Found 11342 variants in bim file.


In [37]:
ht = imputed.s

In [40]:
ht.export("test.txt", header = False)

2022-12-21 09:06:35 Hail: INFO: merging 16 files totalling 3.7M...(15 + 1) / 16]
2022-12-21 09:06:35 Hail: INFO: while writing:
    test.txt
  merge time: 32.686ms


176587

In [18]:
def count_gt_entries(mt: hl.MatrixTable):
    """sum phased/unphased calls 
    :param ht: MatrixTable with gts entry (see collect_gt_by_expr)
    """
    assert 'gts' in list(mt.entry), 'missing entry field "gts"'
   
    return (mt.annotate_entries( 
            hom_alt_n = mt.GT.is_hom_var(),
            phased=hl.struct(
                a1=(mt.GT == hl.parse_call('1|0')),
                a2=(mt.GT == hl.parse_call('0|1'))
            ),
            unphased=hl.struct(
                n=((mt.GT.phased) & (mt.GT.is_het_ref()))
            )
        )
    )

In [16]:
ko.sum_gts_entries(ko.collect_phase_count_by_expr(mt, gene_expr)).describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'gts': array<call>
    'varid': array<str>
    'hom_alt_n': int32
    'phased': struct {
        a1: int32, 
        a2: int32
    }
    'unphased': struct {
        n: int32
    }
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [14]:
ko.aggr_phase_count_by_expr(mt, gene_expr).describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'phased': struct {
        a1: int64, 
        a2: int64, 
        n: int64
    }
    'unphased': struct {
        n: int64
    }
    'hom_alt_n': int64
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [19]:
count_gt_entries(mt, gene_expr).describe()

TypeError: count_gt_entries() takes 1 positional argument but 2 were given

In [7]:
gene_expr = mt.consequence.vep.worst_csq_by_gene_canonical.gene_id
genes = ko.aggr_phase_count_by_expr(mt, gene_expr)
genes = genes.repartition(64)
#expr_pko = ko.calc_prob_ko(genes.hom_alt_n, genes.phased, genes.unphased, only_homs=False)
#expr_ko = ko.annotate_knockout(genes.hom_alt_n, expr_pko)

In [8]:
genes = genes.checkpoint("hail_checkpoint_delete.mt", overwrite = True)

2022-12-07 11:59:16 Hail: INFO: Ordering unsorted dataset with network shuffle2]
2022-12-07 11:59:46 Hail: INFO: wrote matrix table with 142 rows and 176587 columns in 64 partitions to hail_checkpoint_delete.mt
    Total size: 1.85 MiB
    * Rows/entries: 716.36 KiB
    * Columns: 1.15 MiB
    * Globals: 11.00 B
    * Smallest partition: 2 rows (7.86 KiB)
    * Largest partition:  2 rows (22.74 KiB)


In [22]:
genes.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'phased': struct {
        a1: int64, 
        a2: int64, 
        n: int64
    }
    'unphased': struct {
        n: int64
    }
    'hom_alt_n': int64
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [27]:
genes.filter_entries(genes.phased.a1 > 0).show()

+-------------------+---------------------+---------------------+
| gene_id           | '1705888'.phased.a1 | '1705888'.phased.a2 |
+-------------------+---------------------+---------------------+
| str               |               int64 |               int64 |
+-------------------+---------------------+---------------------+
| "ENSG00000141956" |                  NA |                  NA |
| "ENSG00000141959" |                  NA |                  NA |
| "ENSG00000142149" |                  NA |                  NA |
| "ENSG00000142156" |                  NA |                  NA |
| "ENSG00000142166" |                  NA |                  NA |
| "ENSG00000142168" |                  NA |                  NA |
| "ENSG00000142173" |                  NA |                  NA |
| "ENSG00000142182" |                  NA |                  NA |
| "ENSG00000142185" |                  NA |                  NA |
| "ENSG00000142188" |                  NA |                  NA |
+-------------------+---------------------+---------------------+

+--------------------+----------------------+---------------------+
| '1705888'.phased.n | '1705888'.unphased.n | '1705888'.hom_alt_n |
+--------------------+----------------------+---------------------+
|              int64 |                int64 |               int64 |
+--------------------+----------------------+---------------------+
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
+--------------------+----------------------+---------------------+
showing top 10 rows
showing the first 1 of 176587 columns

In [4]:
ht = hl.import_table(phenotypes,
             types={'eid': hl.tstr},
             missing=["",'""',"NA"],
             impute=True,
             force=True,
             key='eid'
             )

2022-11-14 10:47:54 Hail: INFO: Reading table to impute column types(0 + 1) / 1]
2022-11-14 10:48:37 Hail: INFO: Loading <StructExpression of type struct{eid: str, sex: int32, age: int32, `ukbb.centre`: int32, PC1: float64, PC2: float64, PC3: float64, PC4: float64, PC5: float64, PC6: float64, PC7: float64, PC8: float64, PC9: float64, PC10: float64, PC11: float64, PC12: float64, PC13: float64, PC14: float64, PC15: float64, PC16: float64, PC17: float64, PC18: float64, PC19: float64, PC20: float64, PC21: float64, PC22: float64, PC23: float64, PC24: float64, PC25: float64, PC26: float64, PC27: float64, PC28: float64, PC29: float64, PC30: float64, PC31: float64, PC32: float64, PC33: float64, PC34: float64, PC35: float64, PC36: float64, PC37: float64, PC38: float64, PC39: float64, PC40: float64, BC_combined: bool, BC_combined_primary_care: bool, CAD_combined: bool, CAD_combined_primary_care: bool, COPD_combined: bool, COPD_combined_primary_care: bool, CLD_combined: bool, CLD_combined_primary

NameError: name 'mt' is not defined

In [10]:
mt = io.import_table(input_path, input_type)
mt = mt.annotate_cols(pheno=ht[mt.s])

2022-11-14 10:49:32 Hail: INFO: Found 246271 samples in fam file.
2022-11-14 10:49:32 Hail: INFO: Found 16727 variants in bim file.


In [15]:
"CIRR_combined_primary_care" in list(mt.pheno)

True

In [17]:
defined = hl.is_defined(mt.pheno.CIRR_combined_primary_care).collect()

In [11]:
from ukb_utils import variants

In [3]:
p1 = "data/phased/wes_union_calls/200k/shapeit5/phase_rare/ukb_wes_union_calls_shapeit5_200k_chr21-20xshort/shapeit5_prs100000_pro25000_mprs150000.1of1.vcf.gz"
p2 = "data/prephased/wes_union_calls/ukb_wes_union_calls_200k_chr21.vcf.gz"

In [17]:
mt1 = io.import_table(p1, "vcf")
mt2 = io.import_table(p2, "vcf")

In [18]:
mt1 = mt1.annotate_entries(
    GT_rb = mt2[mt1.row_key, mt1.col_key].GT,
    PS_rb = mt2[mt1.row_key, mt1.col_key].PS
)

In [22]:
mt1.transmute_rows(
    info = mt1.info.annotate(
        AC_rb = mt2.rows()[mt1.row_key].info.AC,
        AN_rb = mt2.rows()[mt1.row_key].info.AN
    )
)

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'info': struct {
        AF: float64, 
        AC: int32, 
        AN: int32, 
        AC_rb: int32, 
        AN_rb: int32
    }
----------------------------------------
Entry fields:
    'GT': call
    'PP': float64
    'GT_rb': call
    'PS_rb': int32
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [16]:
mt1.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'info': struct {
        AF: float64, 
        AC: int32, 
        AN: int32
    }
    'AC_rb': int32
    'AN_rb': int32
----------------------------------------
Entry fields:
    'GT': call
    'PP': float64
    'GT_rb': call
    'PS_rb': int32
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [20]:
mt1 = mt1.filter_rows(variants.get_mac_expr(mt1) > 1 )
mt2 = mt2.filter_rows(variants.get_mac_expr(mt2) > 1 )

In [21]:
print(mt1.count())
print(mt2.count())

(10700, 176618)


(10707, 185492)


In [22]:
mt1 = mt1.annotate_rows(mismatch = variants.get_reference_mismatch_expr(mt1.locus, mt1.alleles))
mt2 = mt2.annotate_rows(mismatch = variants.get_reference_mismatch_expr(mt2.locus, mt2.alleles))

In [23]:
print(mt1.aggregate_rows(hl.agg.sum(mt1.mismatch)))

1


In [24]:
print(mt2.aggregate_rows(hl.agg.sum(mt2.mismatch)))

0


In [ ]:
input_path="data/prs/hapmap/ukb_500k/fitting/ukb_hapmap_500k_eur_chrCHR"
input_type="plink"
phenotypes="/well/lindgren/UKBIOBANK/dpalmer/ukb_wes_phenotypes/curated_phenotypes_cts.tsv"
phenotypes="Albumin_residual"
covariates=''
100
data/prs/sumstat/cts/ukb_hapmap_500k_eur_Albumin_residual_chrCHR


In [7]:
input_path = "data/mt/annotated/ukb_eur_wes_union_calls_200k_chr19.mt"

In [8]:
mt = io.import_table(input_path, "mt", calc_info = False)

In [22]:
mt.consequence.vep.worst_csq_for_variant_canonical.most_severe_consequence

<StringExpression of type str>

In [18]:
mt.consequence.vep.worst_csq_by_gene_canonical

<ArrayStructExpression of type array<struct{allele_num: int32, amino_acids: str, biotype: str, canonical: int32, ccds: str, cdna_start: int32, cdna_end: int32, cds_end: int32, cds_start: int32, codons: str, consequence_terms: array<str>, distance: int32, domains: array<struct{db: str, name: str}>, exon: str, gene_id: str, gene_pheno: int32, gene_symbol: str, gene_symbol_source: str, hgnc_id: str, hgvsc: str, hgvsp: str, hgvs_offset: int32, impact: str, intron: str, lof: str, lof_flags: str, lof_filter: str, lof_info: str, minimised: int32, polyphen_prediction: str, polyphen_score: float64, protein_end: int32, protein_start: int32, protein_id: str, sift_prediction: str, strand: int32, swissprot: str, transcript_id: str, trembl: str, uniparc: str, variant_allele: str, revel_score: float64, cadd_phred: float64, most_severe_consequence: str, csq_score: float64}>>

In [3]:
mt = io.import_table("data/unphased/wes_union_calls/bcftools/newtest/tagged.vcf.bgz", "vcf")

In [5]:
mt = mt.annotate_rows(
    missing = mt.aggregate_entries(hl.agg.sum(hl.is_defined(mt.GT)))
)

2022-11-04 15:54:49 Hail: INFO: scanning VCF for sortedness...
2022-11-04 16:00:11 Hail: INFO: Coerced sorted VCF - no additional import work to do


In [6]:
x = mt.missing.collect()

In [10]:
q = [27325787351-y for y in x]